In [ ]:
# -*- coding: utf-8 -*-
"""
T4 GPU Training — SigLIP → SmolLM2 Image Captioning (Cached Embeddings)

Loads pre-computed SigLIP embeddings from siglip_cache.pt, re-tokenizes
captions with SmolLM2, and trains the projector on a frozen SmolLM2-135M
decoder.  No SigLIP forward pass — the expensive part is done once offline.

Usage:
  Run this notebook after precompute_captioner_cache.py.

Requires siglip_cache.pt (generated by precompute_captioner_cache.py).
"""

import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

CACHE_FILE = "siglip_cache.pt"
MODEL_NAME  = "HuggingFaceTB/SmolLM2-135M"

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  1. DEVICE DETECTION                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Hardware] Device: {device}")
if torch.cuda.is_available():
    print(f"[Hardware] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[Hardware] VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


[Hardware] Device: cuda
[Hardware] GPU: Tesla T4
[Hardware] VRAM: 14.6 GB


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  2. MODEL DEFINITION                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class SigLipToTextDecoderV2(nn.Module):
    """Projects SigLIP image embeddings into a frozen SmolLM2 decoder."""

    def __init__(self, llm_model_name: str = MODEL_NAME):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.decoder = AutoModelForCausalLM.from_pretrained(
            llm_model_name,
            torch_dtype=torch.float32,
        )
        for param in self.decoder.parameters():
            param.requires_grad = False

        llm_dim = self.decoder.config.hidden_size
        self.projector = nn.Sequential(
            nn.Linear(768, llm_dim),
            nn.LayerNorm(llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
            nn.LayerNorm(llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, siglip_embeddings, target_token_ids=None):
        siglip_embeddings = siglip_embeddings.to(
            dtype=self.decoder.dtype, device=self.decoder.device
        )
        projected_visual_tokens = self.projector(siglip_embeddings).unsqueeze(1)

        if target_token_ids is not None:
            text_embeddings = self.decoder.model.embed_tokens(target_token_ids)
            full_embeddings = torch.cat(
                [projected_visual_tokens, text_embeddings], dim=1
            )

            ignore_pad = torch.full(
                (target_token_ids.shape[0], 1), -100,
                dtype=target_token_ids.dtype, device=target_token_ids.device,
            )
            padded_labels = torch.cat([ignore_pad, target_token_ids], dim=1)
            padded_labels[padded_labels == self.tokenizer.pad_token_id] = -100

            outputs = self.decoder(
                inputs_embeds=full_embeddings, labels=padded_labels
            )
            return outputs.loss
        else:
            return self.decoder.generate( # type: ignore
                inputs_embeds=projected_visual_tokens, max_new_tokens=40
            )


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  3. LOAD CACHED EMBEDDINGS + DYNAMIC TEXT DATASET                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import random
from torch.utils.data import Dataset

if not os.path.exists(CACHE_FILE):
    print(f"[Error] Cache file '{CACHE_FILE}' not found.")
    sys.exit(1)

print(f"[Data] Loading cached embeddings from '{CACHE_FILE}'...")
cache = torch.load(CACHE_FILE, map_location="cpu", weights_only=True)
embeddings = cache["embeddings"]   # (N, 768) float32
n_samples  = embeddings.shape[0]
print(f"[Data] Loaded {n_samples:,} pre-computed embeddings.")

print("[Data] Loading COCO captions (text only, fast)...")
captions_dataset = load_dataset("phiyodr/coco2017", split="train")

print("[Data] Setup Tokenizer...")
smolm2_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if smolm2_tokenizer.pad_token is None:
    smolm2_tokenizer.pad_token = smolm2_tokenizer.eos_token

# Custom Dataset that pairs each pre-computed embedding with one of the
# 5 available captions, chosen randomly on every access for augmentation.
class CachedEmbeddingsDynamicDataset(Dataset):
    def __init__(self, embeddings_tensor, hf_dataset):
        self.embeddings = embeddings_tensor
        # Load all captions into memory for maximum throughput
        self.captions_list = [item["captions"] for item in hf_dataset]

    def __len__(self):
        return self.embeddings.shape[0]

    def __getitem__(self, idx):
        # Pick a random caption from the 5 available *each time* the
        # dataloader requests this index — built-in data augmentation.
        text = random.choice(self.captions_list[idx])
        return self.embeddings[idx], text

dataset = CachedEmbeddingsDynamicDataset(embeddings, captions_dataset)

# Tokenize on-the-fly using CPU workers
def collate_fn(batch):
    embeds_list = []
    texts = []
    for emb, txt in batch:
        embeds_list.append(emb)
        texts.append(txt)

    batched_embeds = torch.stack(embeds_list)
    token_ids = smolm2_tokenizer(
        texts,
        padding="max_length",
        max_length=40,
        truncation=True,
        return_tensors="pt"
    ).input_ids

    return batched_embeds, token_ids

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  4. MODEL + DATALOADER                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("[Model] Loading SmolLM2 decoder + projector...")
vlm = SigLipToTextDecoderV2().to(torch.float32).to(device)
vlm.train()

# Dynamic tokenization via collate_fn keeps CPU workers busy
dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,  # 2 workers for text-only tokenization
    prefetch_factor=2,
    drop_last=True,
    pin_memory=True,
)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  5. TRAINING LOOP                                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

optimizer = torch.optim.AdamW(vlm.projector.parameters(), lr=5e-4)
scaler = torch.cuda.amp.GradScaler()
epochs = 5

print(f"\n[Training] {n_samples:,} samples, batch_size=64, epochs={epochs}")
print(f"[Training] Device: {device}\n")

for epoch in range(1, epochs + 1):
    pbar = tqdm(dataloader, desc=f"Epoch {epoch:02d}/{epochs}")

    for vision_vectors, tok_ids in pbar:
        optimizer.zero_grad()

        vision_vectors = vision_vectors.to(device)
        tok_ids = tok_ids.to(device)

        # PyTorch 2.x native autocast syntax
        with torch.amp.autocast('cuda', dtype=torch.float16):
            loss = vlm(siglip_embeddings=vision_vectors, target_token_ids=tok_ids)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        # Gradient safety clip
        torch.nn.utils.clip_grad_norm_(vlm.projector.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    print(f"  Epoch {epoch}/{epochs} complete")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  6. SAVE                                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os
os.makedirs("models", exist_ok=True)
vlm.projector.to("cpu")
torch.save(vlm.projector.state_dict(), "models/projector_weights.pth")
print("\n[Done] Projector weights saved to 'models/projector_weights.pth'")